# Solver Choice

How to select an integration method in PaleoBeasts and when that choice has consequences.

## What this does

PaleoBeasts exposes three integration methods: `'RK45'` (adaptive Runge-Kutta via scipy), `'euler'` (fixed-step forward Euler), and `'euler_maruyama'` (stochastic fixed-step). For smooth deterministic ODEs these produce equivalent results given sufficient resolution, with `'RK45'` being far more efficient. For models with **discrete internal state** updated inside `dydt` — where the regime variable is read from and written to `state_variables` on each call — adaptive solvers produce incorrect output because their sub-step evaluations corrupt the regime history. `'euler'` is the only safe choice for those models.

## Models used

- **`Lorenz96`** (n=10, F=8): a smooth chaotic ODE with no internal discrete state. Illustrates how Euler truncation error accumulates exponentially in a chaotic system and why `'RK45'` is the right default for smooth models.
- **`Model3`** (Ganopolski 2024): a piecewise ODE with a discrete regime variable `k` updated inside `dydt`. Illustrates why adaptive solvers cannot be used and how the Euler timestep affects the timing of glacial–interglacial transitions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import paleobeasts as pb
from paleobeasts.signal_models.g24 import calc_f, calc_df

# Shared Lorenz-96 setup
N_L96 = 10
F_L96 = 8.0
T_SPAN_L96 = (0, 20)
Y0_L96 = np.random.default_rng(42).uniform(-1, 1, N_L96)
T_EVAL_L96 = pb.utils.define_t_eval(T_SPAN_L96, delta_t=0.01)

def run_l96(method, dt=None):
    """Fresh Lorenz96 instance integrated with the given method and optional dt."""
    m = pb.Lorenz96(forcing=None, n=N_L96, F=F_L96)
    kw = {'dt': dt} if dt is not None else None
    m.integrate(t_span=T_SPAN_L96, y0=Y0_L96, method=method, kwargs=kw)
    return m.reframe_time_axis(T_EVAL_L96)

# Shared G24 setup
# calc_f(t, A=25, eps=0.5, T1=100, T2=30): synthetic orbital forcing from the paper
# calc_df is its analytical derivative, which Model3 uses for regime-transition detection
G24_FORCING = pb.Forcing(calc_f)
T_SPAN_G24 = (0, 1000)   # kyr
Y0_G24 = [1.0, 1.0]      # v=1.0 (glacial ice volume), k=1 (glacial regime)

def run_g24(dt):
    """Fresh Model3 instance integrated with Euler at the given dt."""
    m = pb.Model3(forcing=G24_FORCING)
    m.integrate(t_span=T_SPAN_G24, y0=Y0_G24, method='euler', kwargs={'dt': dt})
    return m

## Smooth ODE: Lorenz-96

`'RK45'` adapts its step size automatically to keep local error within tolerance, producing accurate trajectories at a fraction of the cost of a sufficiently fine Euler run. For smooth ODEs it is the default. The Lorenz-96 system is chaotic, which means any integration error — including truncation error from a coarse Euler step — grows exponentially. Below, all four runs start from the same initial state. Trajectories diverge from the RK45 reference faster as Euler's dt increases: Euler at dt=0.05 departs noticeably by t≈12, and at dt=0.1 by t≈8.

In [ ]:
state_rk45      = run_l96('RK45')
state_euler_001 = run_l96('euler', dt=0.01)
state_euler_005 = run_l96('euler', dt=0.05)
state_euler_01  = run_l96('euler', dt=0.10)

In [ ]:
runs = [
    (state_rk45,      'RK45 (reference)',  'k',       1.0),
    (state_euler_001, 'Euler  dt = 0.01',  '#1f77b4', 0.9),
    (state_euler_005, 'Euler  dt = 0.05',  '#ff7f0e', 0.9),
    (state_euler_01,  'Euler  dt = 0.10',  '#d62728', 0.9),
]

fig, (ax_traj, ax_err) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for state, label, color, lw in runs:
    ax_traj.plot(T_EVAL_L96, state['x0'], lw=lw, color=color, label=label)
ax_traj.set_ylabel('$x_0$', fontsize=12)
ax_traj.legend(fontsize=9, loc='lower left')
ax_traj.set_title('Lorenz-96 (n=10, F=8): trajectory of $x_0$', fontsize=11)

for state, label, color, lw in runs[1:]:
    err = np.abs(state['x0'] - state_rk45['x0'])
    ax_err.semilogy(T_EVAL_L96, err + 1e-14, lw=lw, color=color, label=label)
ax_err.set_xlabel('time (dimensionless)', fontsize=11)
ax_err.set_ylabel('$|x_0 - x_0^{\\mathrm{RK45}}|$', fontsize=11)
ax_err.legend(fontsize=9, loc='upper left')
ax_err.set_title('Absolute error relative to RK45', fontsize=11)

plt.tight_layout()
plt.show()

## Regime-switching ODE: Ganopolski (2024) Model 3

`Model3` evolves ice volume $v$ through a piecewise ODE whose right-hand side depends on a discrete regime variable $k$ (1 = glacial build-up, 2 = deglaciation). Crucially, `k` is read from and written to `state_variables` inside `dydt` on every call.

**Why `'RK45'` fails here:** an adaptive solver evaluates `dydt` multiple times per step at intermediate times that are not final solution points. Each of these sub-step evaluations modifies `k` in `state_variables`. When the solver evaluates `dydt` at the next "real" timestep, `k` may already have been advanced through one or more transitions by a sub-step call — the regime history is corrupted. The only safe solver is `'euler'`, which calls `dydt` exactly once per timestep in strict chronological order.

**Effect of dt:** even within Euler integration, timestep size matters. If a regime transition happens between two consecutive Euler steps it is missed; only detected at the next step. This shifts the timing of individual glacial cycles. Below, three runs at dt = 1, 4, and 10 kyr show how this plays out over 1000 kyr.

In [ ]:
m_dt1  = run_g24(dt=1)
m_dt4  = run_g24(dt=4)
m_dt10 = run_g24(dt=10)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

for ax, model, label, color in zip(
    axes,
    [m_dt1,  m_dt4,  m_dt10],
    ['Euler  dt = 1 kyr (reference)', 'Euler  dt = 4 kyr', 'Euler  dt = 10 kyr'],
    ['k',    '#1f77b4',  '#d62728'],
):
    t = np.asarray(model.time)
    v = model.state_variables['v']
    k = model.state_variables['k']

    ax.plot(t, v, lw=0.8, color=color, label=label)
    ax.fill_between(
        t, 0, 2.5,
        where=(k == 2),
        alpha=0.15, color='#d62728',
        label='deglaciation (k=2)'
    )
    ax.set_ylim(0, 2.5)
    ax.set_ylabel('ice volume $v$', fontsize=10)
    ax.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('time (kyr)', fontsize=11)
fig.suptitle('Ganopolski (2024) Model 3 — Euler timestep sensitivity', fontsize=12)
plt.tight_layout()
plt.show()

## When to use which solver

| Model type | Solver | dt guidance |
|------------|--------|-------------|
| Smooth, continuous ODE | `'RK45'` | Adaptive; no dt needed |
| Stochastic ODE (SDE) | `'euler_maruyama'` | Model-dependent; dt ≤ 0.01 is a safe starting point |
| Piecewise / regime-switching ODE | `'euler'` | Start at dt = 1/10 of the fastest regime timescale; verify by halving |
| Any model, fast qualitative sweep | `'euler'` | Coarse dt acceptable; do not use output for quantitative comparison |

**Convergence check:** run at dt and dt/2. If the output changes materially (transition timing shifts, amplitude changes), halve again. For `Model3` with default parameters, glacial cycle timing typically converges by dt ≤ 2 kyr.

## See also

- [`lorenz63.ipynb`](lorenz63.ipynb) — signal model notebook for the Lorenz (1963) system
- [`Ganopolski2024_demo.ipynb`](Ganopolski2024_demo.ipynb) — signal model notebook for `Model3` with detailed parameter discussion
- [`melcher2025_do_demo.ipynb`](melcher2025_do_demo.ipynb) — demonstrates `'euler_maruyama'` for stochastic DO events (`Melcher2025DOModel`)